## CS336 playground

In [2]:
import time
import torch
import os
import cs336_basics.ron_adamw_optimizer as ron_adamw_optimizer
import cs336_basics.ron_bpe_tokenizer as ron_bpe_tokenizer
import cs336_basics.ron_causal_multihead_self_attention_with_rope as ron_causal_multihead_self_attention_with_rope
import cs336_basics.ron_cross_entropy as ron_cross_entropy
import cs336_basics.ron_data_loader as ron_data_loader
import cs336_basics.ron_embedding as ron_embedding
import cs336_basics.ron_linear as ron_linear
import cs336_basics.ron_multihead_self_attention as ron_multihead_self_attention
import cs336_basics.ron_rmsnorm as ron_rmsnorm
import cs336_basics.ron_rope as ron_rope
import cs336_basics.ron_scaled_dot_product_attention as ron_scaled_dot_product_attention
import cs336_basics.ron_softmax as ron_softmax
import cs336_basics.ron_swiglu as ron_swiglu
import cs336_basics.ron_train_bpe as ron_train_bpe
import cs336_basics.ron_transformer_lm as ron_transformer_lm

In [3]:
# from the 7.2 secion

vocab_size = 10000
context_length = 256
d_model = 512
d_ff = 1344
rope_theta = 10000
num_layers = 4
num_heads = 16
batch_size = 16 # fits in 6GB Nvidia 2060
batch_size = 32
device = (
    (torch.cuda.is_available() and 'cuda') or
    (torch.backends.mps.is_available() and 'mps') or
    "cpu"
)

!wget -q -nc https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt -O /tmp/tiny_shakespeare.txt
!wget -q -nc https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-train.txt -O ../data/TinyStoriesV2-GPT4-train.txt
!wget -q -nc https://dgoldberg.sdsu.edu/515/harrypotter.txt -O /tmp/harrypotter.txt

datasets = {
   'tinystories'            : '../data/TinyStoriesV2-GPT4-train.txt',
   'shakespeare'            : '/tmp/tiny_shakespeare.txt',
   'book'                   : '/tmp/harrypotter.txt',
   'tinystories_validation' : '../data/TinyStoriesV2-GPT4-valid.txt',
}
vocab_dataset = 'shakespeare'
training_dataset = 'shakespeare'


## BPE training

In [4]:
"""
    vocab size of 1000 on TinyStoriesV2-GPT4-valid.txt  
    isn't horrible for CPU-based interactive notebook use.
    It takes 16 seconds to train and makes words like " Sally" and " helped" 
    into single tokens. Maybe roughly the vocab of a preschooler :)

    vocab size of 10000 on TinyStoriesV2-GPT4-valid.txt takes like 
    three minutes. and makes words like " sinking" and " ribbit".

    Shakespeare's about 3 minutes as well. 
"""
def get_vocab_and_merges(dataset='tinystories',vocab_size=vocab_size):
    vocab_cache = f"/tmp/bpe_{dataset}_{vocab_size}.saved"
    if not os.path.exists(vocab_cache):
        vocab_source = datasets[dataset]
        vocab,merges = ron_train_bpe.train_bpe(vocab_source,vocab_size,[])
        obj = {"vocab": vocab,"merges": merges}
        torch.save(obj, vocab_cache)
    else:
        obj = torch.load(vocab_cache)
        vocab,merges = obj['vocab'],obj['merges']
    return vocab,merges
print(vocab_dataset)
vocab,merges = get_vocab_and_merges(vocab_dataset)
print(merges[-10:])

shakespeare
[(b'read', b'ing'), (b're', b'qu'), (b're', b'cy'), (b're', b'ck'), (b're', b'b'), (b're', b'aming'), (b're', b'am'), (b'ray', b'be'), (b'rant', b's'), (b'ran', b'g')]


## BPE class

In [5]:
tokenizer = ron_bpe_tokenizer.RonBPETokenizer(vocab,merges)
tokens    = list(tokenizer.encode_iterable(["Hello world"," ","Good bye"]))
print(tokens)
print(tokenizer.decode(tokens))

[72, 408, 111, 864, 32, 1227, 415, 101]
Hello world Good bye


### Make a nice training dataset with that tokenizer

In [6]:
import numpy as np
import tqdm

def get_tokenized_dataset(dataset, vocab_dataset, vocab_size, dtype=np.uint16):
    out_path = f"/tmp/{dataset}_{vocab_dataset}_{vocab_size}.uint16"
    if not os.path.exists(out_path):
        print("creating ", out_path, " please be patient...")
        tokenizer = ron_bpe_tokenizer.RonBPETokenizer(*get_vocab_and_merges(vocab_dataset,vocab_size))
        data_source = datasets[dataset]
        with open(out_path, "ab") as fout:
            with open(data_source, "r", encoding="utf-8",errors="ignore") as fin:
                for line in tqdm.tqdm(fin):
                    toks = tokenizer.encode(line + "\n")
                    toks_np = np.asarray(toks, dtype=dtype)
                    fout.write(toks_np.tobytes())
    print(f"using {out_path}")
    token_ds = np.memmap(out_path, dtype=np.uint16, mode="r")
    return token_ds

token_ds = get_tokenized_dataset(training_dataset, vocab_dataset, vocab_size)
token_ds[0:10]

using /tmp/shakespeare_shakespeare_10000.uint16


memmap([ 672, 1196,   58,   10,   10, 2323,  331, 2736,  803, 2294],
       dtype=uint16)

## My Transformer model


In [7]:
import cs336_basics.ron_transformer_lm as ron_transformer_lm

tlm = ron_transformer_lm.TransformerLM(
    d_model=d_model,
    num_heads=num_heads,
    d_ff = d_ff,
    max_seq_len=context_length,
    theta=rope_theta,
    vocab_size=vocab_size,
    context_length=context_length,
    num_layers=num_layers
    )
tlm.to(device)
inputs = torch.tensor([tokenizer.encode("Hello world")])
tlm.forward(inputs)

/Users/rmayer@shotspotter.com/work/cs336/assignment1-basics/.venv/lib/python3.13/site-packages/torch/_tensor_str.py:145: UserWarning: MPS: nonzero op is supported natively starting from macOS 14.0. Falling back on CPU. This may have performance implications. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/mps/operations/Indexing.mm:401.)
  nonzero_finite_vals = torch.masked_select(


tensor([[[-0.2501,  0.3282,  0.4392,  ..., -0.4316,  0.0209, -0.0539],
         [ 0.0324, -0.2165,  0.4750,  ..., -0.1804, -0.3124,  0.0937],
         [-0.3181,  0.6615,  0.1244,  ..., -0.1964, -0.1215,  0.1267],
         [-0.4464,  0.4497, -0.2819,  ..., -0.8271,  0.2173, -0.3925]]],
       device='mps:0', grad_fn=<ViewBackward0>)

### Predict words

* Expected to be quite random here - no training occurred

In [9]:
def predict_words(
        tlm: ron_transformer_lm.TransformerLM,
        tokenizer: ron_bpe_tokenizer.RonBPETokenizer,
        num_tokens_to_generate = 50,
        initial_text: str = "The",
):
    generated_tokens = tokenizer.encode(initial_text)
    for _ in range(num_tokens_to_generate):
        seq_len = len(generated_tokens)
        tokens_tensor = torch.tensor([generated_tokens])
        with torch.no_grad():
            predictions = tlm.forward(tokens_tensor)  # (1, seq_len, vocab_size)
        last_logits = predictions[0, -1]
        #next_token_id = last_logits.argmax().item()
        probs = ron_softmax.softmax(last_logits,-1)
        next_token_id = torch.multinomial(probs, num_samples=1).item()
        generated_tokens.append(next_token_id)

    return tokenizer.decode(generated_tokens)

predict_words(tlm,tokenizer,20)

'Theremyr TimetinoISABEL Welcomeuckoldton superfl calf maystinks Answer Fortune manage clergyWISWomen protZABET'

## My AdamW Optimizer

In [10]:
my_adamw = ron_adamw_optimizer.AdamW(tlm.parameters())

## My Data Loader

In [11]:
def get_training_batch():
    x,y = ron_data_loader.get_batch(token_ds,batch_size,context_length,device)
    if device == 'mps':
        # TypeError: Trying to convert UInt16 to the MPS backend but it does not have support for that dtype.
        x = x.to('cpu').to(dtype=torch.long).to(device)
        y = y.to('cpu').to(dtype=torch.long).to(device)
    else:
        x = x.to(dtype=torch.long)
        y = y.to(dtype=torch.long)
    return x,y


### Compile the model

Can take 14 seconds for the first forward pass...

Quoting a chatbot: 

> PyTorch (via TorchDynamo + TorchInductor) traces your model the first time you call it, captures the computation graph, and then compiles optimized kernels for your GPU.
>
> What happens:
>
> * First call (slow, e.g. 14s)
>   * TorchDynamo intercepts your Python code.
>   * TorchInductor generates optimized code (usually CUDA kernels).
>   * Compilation is JIT (just-in-time), so it can take several seconds.
> * Subsequent calls (fast, e.g. 0.2s)
>   * The compiled kernels are cached.
>   * PyTorch skips Python dispatch and just runs the optimized kernels.
>
> That’s why you see a huge difference between the first iteration and the rest.

This warning is expected on my cheap old GPU

```
W0901 17:10:39.072000 3071772 torch/_inductor/utils.py:1137] [0/0] Not enough SMs to use max_autotune_gemm mode
```




In [12]:
# warmup to not skew training iter time
if device != 'mps': # fails on 'mps'
    tlm.compile()
    tokens = torch.randint(0, vocab_size, (16, 256), device=device)
    tlm(tokens)


## Training Loop

In [ ]:
import mlflow
import IPython.display as ipd
import html
import regex as re
def train(model, optimizer, get_batch, device=device, max_training_time = 60 * 5, num_iters=1000*1000):
    model.to(device)
    model.train()

    with mlflow.start_run():
        mlflow.log_params({
            "device": device,
            "num_iters": num_iters,
            "optimizer": type(optimizer).__name__,
            "model": type(model).__name__
        })
        t0 = time.time()
        next_log_interval = 1
        for it in range(num_iters):
            x, y = get_batch()
            logits = model(x)            # shape: (batch, seq, vocab)
            loss = ron_cross_entropy.cross_entropy(
                logits.view(-1, logits.size(-1)),
                y.view(-1)
            )
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if it >= next_log_interval or time.time() > max_training_time + t0:
                next_log_interval *= 2
                mlflow.log_metric("loss", loss.item(), step=it)
                mlflow.log_metric("elapsed_time", time.time() - t0, step=it)
                output = predict_words(tlm,tokenizer,100)
                ipd.display(ipd.HTML(f'''
                    <div style="display: grid; grid-template-columns: 1fr 3fr; gap: 1px; border: 1px solid black; padding: 1px;">
                        <div style="border-right: 1px solid black; padding: 5px;">
                            <strong>iter {it}</strong><br>
                            loss: {loss.item():.4f}<br>
                            elapsed: {time.time() - t0:.3f} secs
                        </div>
                        <div style="padding: 1px;">
                            {re.sub("\n+", "<br>", html.escape(output))}
                        </div>
                    </div>
                '''), metadata=dict(isolated=True))
                if time.time() - t0 > max_training_time:
                    break

In [14]:
train(tlm,my_adamw,get_training_batch,max_training_time=60*2)

## Or an example trained on tinystories

* Training loop iteration 1 - it's basically random broken words
* Training loop iteration 8 - some idea of adjectives and verbs, but nonsense sentences.
* Training loop iteration 32 - some concept of grammar, forming.
* Training loop iteration 256 - forming coherent sentences with surreal imagery; but no flow between sentences.
* Training loop iteration 1024 - pairs of sentences seem connected; but no overarching consistent theme.
* Training loop iteration 2048 - a consistent theme throughout the paragraph.
* Training loop iteration 4096 - more interesting settings and character interactions
* Training loop iteration 8192 - characters gain empathy toward each other
* Training loop iteration 16384 - characters gain empathy toward each society

Neat to see these different emergent concepts develop!

In [ ]:
train(tlm,my_adamw,get_training_batch,max_training_time=60*30)